# Volume 2, Chapter 4: The Symmetric Group (Python companion)

Exercises first, solutions at the end. We compute Landau's function, check the class sizes of $\mathfrak S_n$ and $A_n$, solve the $8$-puzzle completely by breadth-first search, compute the order of the Rubik's cube group, and write the algorithm that makes such computations possible: **Schreier–Sims**, which finds the order of a permutation group of size $4\times10^{19}$ in a fraction of a second, using nothing but the orbit–stabiliser theorem.

*Before each solution, predict what the output will look like.*

In [ ]:
import itertools
import math
import time
from collections import Counter, deque
import numpy as np
import matplotlib.pyplot as plt
from sympy.combinatorics import Permutation, PermutationGroup
from sympy.combinatorics.named_groups import RubikGroup
from sympy.utilities.iterables import partitions

## Exercises

**P4.1 (★) Landau's function.** Compute $g(n)$, the largest order of an element of $\mathfrak S_n$, for $n\le 60$: by brute force over the partitions of $n$ for $n\le 25$, and by the reduction of Exercise 4.1 (powers of distinct primes with sum $\le n$) up to $60$. Compare $\log g(n)$ with $\sqrt{n\log n}$.

In [ ]:
# Your code here


**P4.2 (★) Class sizes.** Check that the class sizes $n!/z_\lambda$ of Proposition 4.4 add up to $n!$ for $n\le 12$. Then predict, with the criterion of Exercise 4.8, the number of conjugacy classes of $A_n$ for $n\le 12$, and check $n\le 6$ by brute force. *Guess first: how many classes has $A_5$?*

In [ ]:
# Your code here


**P4.3 (★★) The 8-puzzle.** On the $3\times3$ board, explore all positions reachable from the solved one by breadth-first search. Check that there are $9!/2=181440$ of them and that they are exactly those with $\varepsilon\delta=1$ (Theorem 4.7, which holds on this board with the same proof). Plot how many positions lie at each distance from the solved one. What is the hardest position?

In [ ]:
# Your code here


**P4.4 (★★) Rubik's cube.** SymPy's `RubikGroup(3)` is the group generated by the six face turns, acting on the $54$ stickers. Find its order, check that it equals $8!\cdot3^8\cdot12!\cdot2^{12}/12$ (Exercise 4.6), find the cycle type of a generator on the stickers, and the orders of the products of two generators.

In [ ]:
# Your code here


**P4.5 (★★★) CS application: the Schreier–Sims algorithm.** A permutation group given by a few generators can be astronomically large, so it cannot be listed. Schreier–Sims computes its order through a chain of stabilisers $G=G_0\ge G_1\ge\dots$, where $G_{i+1}$ fixes one more point $b_i$; by orbit–stabiliser, $|G|=\prod_i|G_i\cdot b_i|$. The difficulty is to find generators of each $G_{i+1}$: Schreier's lemma says that if $T$ is a set of coset representatives ($u_x\cdot b=x$) then the elements $u_{s(x)}^{-1}\,s\,u_x$ ($s$ a generator, $x$ in the orbit) generate the stabiliser.
1. Implement it, and check it on $\mathfrak S_8$, $A_8$, the rotations of the cube and the Rubik's cube group.
2. Use it to confirm the converse half of Theorem 4.7 on the $4\times4$ board: the seven rectangle loops of its proof generate a group of order $15!/2$.
3. Perfect shuffles (Exercise 4.7): find the orders of the out-shuffle and the in-shuffle on $52$ cards, and the order of the group they generate together. Explain why it is so much smaller than $52!$.

In [ ]:
# Your code here


---
## Solutions

### P4.1

In [ ]:
def g_brute(n):
    return max(math.lcm(*[k for k, m in p.items() for _ in range(m)]) for p in partitions(n))

def primes_upto(n):
    return [p for p in range(2, n + 1) if all(p % d for d in range(2, int(p**0.5) + 1))]

def g_fast(n):
    # best[s] = largest product of powers of distinct primes (processed so far) with total sum exactly s
    best = [1] + [0] * n
    for p in primes_upto(n):
        new = best[:]
        q = p
        while q <= n:
            for s in range(n - q, -1, -1):
                if best[s] and best[s] * q > new[s + q]:
                    new[s + q] = best[s] * q
            q *= p
        best = new
    return max(best)

print([g_brute(n) for n in range(1, 26)] == [g_fast(n) for n in range(1, 26)])
print([g_fast(n) for n in range(1, 21)])
ns = np.arange(2, 61)
g = np.array([g_fast(n) for n in ns], dtype=float)
plt.figure(figsize=(6, 3.2))
plt.plot(ns, np.log(g), 'o', ms=3, label='log g(n)')
plt.plot(ns, np.sqrt(ns * np.log(ns)), label='sqrt(n log n)')
plt.xlabel('n'); plt.legend(); plt.tight_layout(); plt.show()
print('g(60) =', g_fast(60))

Both methods agree, and $g(7)=12$, $g(10)=30$ as in Exercise 4.1. The table needs every *exact* sum $s\le n$, padded with fixed points afterwards, which is why we take the maximum over all $s$. The ratio $\log g(n)/\sqrt{n\log n}$ tends to $1$ (Landau, 1903), but very slowly: the plot shows $\log g(n)$ well below its asymptotic guess at $n=60$, with a staircase shape, because $g$ only jumps when a new prime fits.

### P4.2

In [ ]:
def z(p):
    return math.prod(k**m * math.factorial(m) for k, m in p.items())

for n in range(1, 13):
    assert sum(math.factorial(n) // z(p) for p in partitions(n)) == math.factorial(n)
print('class sizes add up to n! for n <= 12')

def classes_An(n):
    count = 0
    for p in partitions(n):
        parts = [k for k, m in p.items() for _ in range(m)]
        if sum(k - 1 for k in parts) % 2:          # odd type: not in A_n
            continue
        splits = all(k % 2 for k in parts) and all(m == 1 for m in p.values())
        count += 2 if splits else 1
    return count

def classes_brute(n):
    G = [Permutation(q) for q in itertools.permutations(range(n))]
    A = [x for x in G if x.is_even]
    left, k = set(A), 0
    while left:
        x = left.pop(); k += 1
        left -= {y * x * y**-1 for y in A}
    return k

print('A_n classes (criterion):', [classes_An(n) for n in range(2, 13)])
print('A_n classes (brute)    :', [classes_brute(n) for n in range(2, 7)])
A5 = [x for x in (Permutation(q) for q in itertools.permutations(range(5))) if x.is_even]
sizes, left = [], set(A5)
while left:
    x = left.pop(); cl = {y * x * y**-1 for y in A5}; sizes.append(len(cl)); left -= cl
print('A5 class sizes:', sorted(sizes))

The criterion and brute force agree; $A_5$ has $5$ classes of sizes $1,12,12,15,20$, as in Theorem 4.9. For $n=2$ the group $A_2$ is trivial, with one class, and the criterion gives $1$ (the type $1^2$ repeats a part). Notice $A_4$: $4$ classes, because the $3$-cycles split.

### P4.3

In [ ]:
N = 3
solved = tuple(range(1, N * N)) + (0,)          # 0 is the blank, cells in reading order

def neighbours(pos):
    b = pos.index(0); r, c = divmod(b, N)
    for dr, dc in ((1, 0), (-1, 0), (0, 1), (0, -1)):
        rr, cc = r + dr, c + dc
        if 0 <= rr < N and 0 <= cc < N:
            k = rr * N + cc
            q = list(pos); q[b], q[k] = q[k], q[b]
            yield tuple(q)

t = time.time()
dist = {solved: 0}
queue = deque([solved])
while queue:
    p = queue.popleft()
    for q in neighbours(p):
        if q not in dist:
            dist[q] = dist[p] + 1; queue.append(q)
print(len(dist), math.factorial(9) // 2, f'({time.time() - t:.1f} s)')

def invariant(pos):
    # sigma(label) = cell, with the blank labelled N*N; delta = +1 iff blank on a cell of the corner's colour
    sigma = [0] * (N * N)
    for cell, lab in enumerate(pos):
        sigma[(lab or N * N) - 1] = cell
    eps = Permutation(sigma).signature()
    b = pos.index(0); r, c = divmod(b, N)
    delta = 1 if (r + c) % 2 == (2 * (N - 1)) % 2 else -1
    return eps * delta

print('all reachable have invariant +1:', all(invariant(p) == 1 for p in dist))
print('unreachable example:', invariant((1, 2, 3, 4, 5, 6, 8, 7, 0)))
hist = Counter(dist.values())
D = max(hist)
plt.figure(figsize=(6, 3)); plt.bar(list(hist), [hist[d] for d in hist])
plt.xlabel('distance from solved'); plt.ylabel('positions'); plt.tight_layout(); plt.show()
print('maximal distance:', D, ' positions at that distance:', [p for p in dist if dist[p] == D])

Exactly $181440=9!/2$ positions are reachable, all with $\varepsilon\delta=1$, and swapping $7$ and $8$ gives an unreachable one. The distance histogram is bell-shaped, and the hardest positions need $31$ moves: there are exactly two of them. Breadth-first search is possible here because the state space is small; for the $4\times4$ puzzle there are $16!/2\approx10^{13}$ states, and the exact maximum distance, $80$ moves, was established by a complete breadth-first search stored on disk (Korf and Schultze, 2005). For the Rubik's cube, the analogous number is $20$ (Rokicki et al., in the sources).

### P4.4

In [ ]:
G = RubikGroup(3)
order = G.order()
print(order, order == math.factorial(8) * 3**8 * math.factorial(12) * 2**12 // 12)
print('factorised:', __import__('sympy').factorint(order))
F = G.generators[0]
print('cycle type of a face turn on the 54 stickers:', F.cycle_structure)
gens = G.generators
print('orders of the six generators:', [g.order() for g in gens])
print('orders of products of two generators:')
for a in gens:
    print([(a * b).order() for b in gens])

A face turn moves $20$ stickers in five $4$-cycles: the $8$ non-centre stickers of the face (two $4$-cycles) and the $12$ stickers around it (three $4$-cycles). The order is $2^{27}3^{14}5^37^211$, exactly $8!\cdot3^8\cdot12!\cdot2^{12}/12$. The products of two generators have small orders, the largest being $105$: this is the order of the classic sequence $RU$ (turn the right face, then the top face), which brings the cube back after $105$ repetitions. Small orders like these are typical: by Corollary 4.3 the order of a move is the lcm of its cycle lengths on the $48$ moving stickers, and the largest order of any element of the cube group is $1260$.

### P4.5

In [ ]:
def mul(p, q):                    # p∘q: first q, then p
    return tuple(p[i] for i in q)

def inv(p):
    r = [0] * len(p)
    for i, x in enumerate(p):
        r[x] = i
    return tuple(r)

def schreier_sims(gens):
    n = len(gens[0]); e = tuple(range(n))
    base, S, T = [], [], []       # S[i] generates G_i; T[i] maps each point x of the orbit of base[i] to some u with u(base[i]) = x
    def orbit(i):
        T[i] = {base[i]: e}; todo = [base[i]]
        while todo:
            x = todo.pop()
            for s in S[i]:
                y = s[x]
                if y not in T[i]:
                    T[i][y] = mul(s, T[i][x]); todo.append(y)
    def sift(g):                  # divide g by coset representatives, level by level
        for i in range(len(base)):
            b = g[base[i]]
            if b not in T[i]:
                return g, i
            g = mul(inv(T[i][b]), g)
        return g, len(base)
    def insert(g, k):
        if k == len(base):
            base.append(next(x for x in range(n) if g[x] != x)); S.append([]); T.append({})
        for j in range(k + 1):
            S[j].append(g); orbit(j)
    for g in gens:
        r, k = sift(g)
        if r != e:
            insert(r, k)
    checked = set()
    while True:                   # until every Schreier generator sifts to the identity
        new = None
        for j in reversed(range(len(base))):
            for x, u in list(T[j].items()):
                for s in S[j]:
                    if (j, u, s) in checked:
                        continue
                    r, k = sift(mul(inv(T[j][s[x]]), mul(s, u)))
                    if r != e:
                        new = (r, k); break
                    checked.add((j, u, s))
                if new: break
            if new: break
        if new is None:
            return math.prod(len(t) for t in T), base
        insert(*new)

def cyc(n, *cycles):
    p = list(range(n))
    for c in cycles:
        for a, b in zip(c, c[1:] + c[:1]):
            p[a] = b
    return tuple(p)

tests = {
    'S8': [cyc(8, (0, 1)), cyc(8, tuple(range(8)))],
    'A8': [cyc(8, (0, 1, 2)), cyc(8, tuple(range(1, 8)))],
    'cube rotations on faces': [(0, 1, 5, 4, 2, 3), (3, 2, 0, 1, 4, 5)],
    'Rubik': [tuple(g.array_form) for g in RubikGroup(3).generators],
}
for name, gens in tests.items():
    t = time.time(); o, b = schreier_sims(gens)
    print(f'{name:24s} order {o}  (base of length {len(b)}, {time.time() - t:.2f} s)')

In [ ]:
# 2. The rectangle loops of the proof of Theorem 4.7, as permutations of the cells 1..15 (indices 0..14)
def loop(top, left):
    cell = lambda r, c: 4 * (r - 1) + c
    path = ([cell(4, c) for c in range(4, left - 1, -1)] + [cell(r, left) for r in range(3, top - 1, -1)]
            + [cell(top, c) for c in range(left + 1, 5)] + [cell(r, 4) for r in range(top + 1, 5)])
    content = {x: x for x in range(1, 17)}; blank = 16
    for nxt in path[1:]:
        content[blank], content[nxt] = content[nxt], content[blank]; blank = nxt
    h = [0] * 15
    for c in range(1, 16):
        h[content[c] - 1] = c - 1       # the tile that started in cell content[c] is now in cell c
    return tuple(h)

loops = [loop(t, l) for t, l in [(3, 3), (3, 2), (3, 1), (2, 3), (1, 3), (2, 1), (1, 1)]]
o, _ = schreier_sims(loops)
print(o, math.factorial(15) // 2, o == math.factorial(15) // 2)
print('the 2x2 loop alone:', schreier_sims(loops[:1])[0], '  without the 2x2 loop:', schreier_sims(loops[1:])[0])

In [ ]:
# 3. Perfect shuffles on 52 cards: position i -> new position
out_sh = tuple(51 if i == 51 else 2 * i % 51 for i in range(52))
in_sh = tuple(2 * (i + 1) % 53 - 1 for i in range(52))
def order(p):
    k, q = 1, p
    while q != tuple(range(len(p))):
        q = mul(p, q); k += 1
    return k
print('out-shuffle order', order(out_sh), ' in-shuffle order', order(in_sh))
o, _ = schreier_sims([out_sh, in_sh])
print('group generated by both:', o)
mirror = tuple(51 - i for i in range(52))
print('both shuffles commute with i -> 51-i:', mul(out_sh, mirror) == mul(mirror, out_sh), mul(in_sh, mirror) == mul(mirror, in_sh))
bound = 2**26 * math.factorial(26)
print('order equals 2^26 * 26!:', o == bound, '  52!/order = %.3e' % (math.factorial(52) / o))

Schreier–Sims returns the orders $8!$, $8!/2$, $24$ and $43\,252\,003\,274\,489\,856\,000$, the last in well under a second, while listing the group would take longer than the age of the universe. The algorithm is the orbit–stabiliser theorem applied repeatedly, with Schreier's lemma to produce generators of each stabiliser, and the *sift* procedure (dividing by coset representatives level by level) to test membership. It is the engine inside GAP, Magma and SymPy's `PermutationGroup.order()`.

The seven rectangle loops of the proof of Theorem 4.7 generate a group of order $15!/2$, as the proof shows; the $2\times2$ loop alone gives only a $3$-cycle, of order $3$.

The out-shuffle has order $8$ and the in-shuffle order $52$, as in Exercise 4.7. Together they generate a group much smaller than $52!$: both shuffles keep the cards in positions $i$ and $51-i$ mirror images of each other (they commute with $i\mapsto51-i$), so the group permutes the $26$ mirror pairs and can swap the two cards of each pair; it lies in the group of all such permutations, of order $2^{26}\cdot26!$, and the computation shows it is all of it. Diaconis, Graham and Kantor (sources of Chapter 4) determined this group for every deck size; magicians use exactly this mirror property.